In [2]:
import logging
import time
import os
import pickle


import numpy as np
import matplotlib.pyplot as plt

#import tensorflow_datasets as tfds
import tensorflow as tf

# Import tf_text to load the ops used by the tokenizer saved model
#import tensorflow_text  # pylint: disable=unused-import
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib as plt


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model,  Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dropout, Input, TimeDistributed, Dense, Activation, RepeatVector, Embedding, Concatenate
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Attention
from tensorflow.keras.optimizers import Adam, Adagrad
from keras.losses import sparse_categorical_crossentropy
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings
import random

In [3]:
# data preprocessing functions - pulled from DataPrep_AllData.py
def tokenize_AA(sequences):
    AA_dict = {'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10,
               'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19,
               'Y': 20, 'X': 21, 'Z': 21, 'B': 21, 'U': 21, 'O': 21, '*': 22}

    seq_tokenized = []
    for s in range(len(sequences)):
        seq = sequences[s]
        temp = [24]  # Start token
        for i in range(len(seq)):
            temp.append(AA_dict[seq[i]])
        temp.append(23)

        seq_tokenized.append(temp)

    return seq_tokenized, AA_dict




def AA_Codon_list():
    dic_AA_codon = {'A': ['GCT', 'GCC', 'GCA', 'GCG'],
                    'C': ['TGT', 'TGC'],
                    'D': ['GAT', 'GAC'],
                    'E': ['GAA', 'GAG'],
                    'F': ['TTT', 'TTC'],
                    'G': ['GGT', 'GGA', 'GGC', 'GGG'],
                    'H': ['CAT', 'CAC'],
                    'I': ['ATT', 'ATC', 'ATA'],
                    'K': ['AAA', 'AAG'],
                    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
                    'M': ['ATG'],
                    'N': ['AAT', 'AAC'],
                    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
                    'Q': ['CAA', 'CAG'],
                    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
                    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
                    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
                    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
                    'W': ['TGG'],
                    'Y': ['TAT', 'TAC'],
                    '*': ['TAA', 'TAG', 'TGA']}

    codon_list = []
    AA_list = []
    for key in dic_AA_codon:
        for i in range(len(dic_AA_codon[key])):
            AA_list.append(key)
        for i in dic_AA_codon[key]:
            codon_list.append(i)
    return AA_list, codon_list

def tokenize_Codon(sequences):
    AA_list, codon_list = AA_Codon_list()
    keys = codon_list
    values = range(1, len(codon_list) + 1)
    Codon_dict = dict(zip(keys, values))
    seq_tokenized = []
    for s in range(len(sequences)):
        seq = sequences[s]
        temp = [65]  # Start token
        for i in range(int(len(seq) / 3)):
            temp.append(Codon_dict[seq[3 * i: 3 * (i + 1)]])
        
        temp.append(66)
        seq_tokenized.append(temp)

    return seq_tokenized, Codon_dict

def data_prep(N_organisms, Data_dict): #
    Data_dict_New = {}
    for o in range(N_organisms):
        index = []
        for i in range(len(Data_dict[o]['CDS_Seq'])):
            if Data_dict[o]['CDS_Seq'][i].find('N') == -1:
                index.append(i)

        Data_dict_New[o] = Data_dict[o].iloc[index, :]

    AA_Seq_org = {}
    CDS_Seq_org = {}
    AA_Seq_list = []
    CDS_Seq_list = []

    for o in range(N_organisms):
        AA_Seq_org[o] = Data_dict_New[o]['AA_Seq'].tolist()
        CDS_Seq_org[o] = Data_dict_New[o]['CDS_Seq'].tolist()
        AA_Seq_list = AA_Seq_list + AA_Seq_org[o]
        CDS_Seq_list = CDS_Seq_list + CDS_Seq_org[o]

    Nreps = 25
    AA_list, codon_list = AA_Codon_list()
    AA_list_rep = AA_list
    codon_list_rep = codon_list

    for k in range(Nreps):
        AA_list_rep = AA_list_rep + AA_list
        codon_list_rep = codon_list_rep + codon_list

    AA_Seq_list = AA_list_rep + AA_Seq_list
    CDS_Seq_list = codon_list_rep + CDS_Seq_list

    # temp = list(zip(AA_Seq_list, CDS_Seq_list))
    # random.shuffle(temp)
    # res1, res2 = zip(*temp)
    # # res1 and res2 come out as tuples, and so must be converted to lists.
    # AA_Seq_list_shuffle, CDS_Seq_list_shuffle = list(res1), list(res2)

    AA_seq_tokenized, AA_seq_tokenizer = tokenize_AA(AA_Seq_list)
    Cds_seq_tokenized, Cds_seq_tokenizer = tokenize_Codon(CDS_Seq_list)

    AA_pad_seq = pad_sequences(AA_seq_tokenized, maxlen=10001, dtype='int32', padding="post", truncating="post")
    Cds_pad_seq = pad_sequences(Cds_seq_tokenized, maxlen=1000, dtype='int32', padding="post", truncating="post")

    #AA_tr, AA_ts, Cds_tr, Cds_ts = train_test_split(AA_pad_seq, Cds_pad_seq, test_size=0.10, random_state=42)
    AA_tr = AA_pad_seq[len(AA_list):, :]
    Cds_tr = Cds_pad_seq[len(codon_list):, :]
    AA_ts = AA_pad_seq[0:len(AA_list), :]
    Cds_ts = Cds_pad_seq[0:len(codon_list), :]

    return AA_tr, Cds_tr, AA_ts, Cds_ts

In [4]:
# this pulls some pre-processed data that wasn't included in the repo, need to do this processing
# need to update these file references
N_organisms = 5
Data_dict = {0:pd.read_csv('/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/CBS7435.csv'),  
             1:pd.read_csv('/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/GS115_ext.csv'), 
             2:pd.read_csv('/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/Kpastoris_WT.csv'), 
             3:pd.read_csv('/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/Kphaffi_GS115_int.csv'), 
             4:pd.read_csv('/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/Kphaffi_WT.csv')}

AA_tr,  Cds_tr, AA_ts, Cds_ts = data_prep(N_organisms, Data_dict)

Data_prepped = {'AA_tr': AA_tr, 
               'AA_ts': AA_ts, 
                'Cds_tr': Cds_tr,
                'Cds_ts':Cds_ts}

In [6]:
# write this prepped data to a .pkl file for future import
out_pkl = '/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/Pichia_All_2Target.pkl'
with open(out_pkl, 'wb') as output:
    # Pickle dictionary using protocol 0.
    pickle.dump(Data_prepped, output)

In [7]:
# for future trainings, can start here and load the pre-processed data already
out_pkl = '/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/AllData/Pichia_All_2Target.pkl'
parameter_settings_csv = "/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/Training/BO_forHyperParameter/Arch1/Round3.csv"


with open(out_pkl, "rb") as fp:
    Data_AllOrg = pickle.load(fp)
    
AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']
AA_ts = Data_AllOrg['AA_ts']
Cds_ts = Data_AllOrg['Cds_ts']

Settings = pd.read_csv(parameter_settings_csv).iloc[:, 1:]
Setting_no = 1


    
Max_length = 1000
learning_rate = 0.001
batch_size = 150
epochs = 100
aa_vocab_size = 25
dna_vocab_size = 67


hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])

drop_rate = Settings['Drop rate'][Setting_no]
drop_rate_aa = Settings['Drop rate aa'][Setting_no]


In [8]:
Settings

,Enc hidden size,Enc Embedding size,Dec Embedding size,Dense Layer size,Dense Layer size aa,Drop rate,Drop rate aa
0,506.0,100.0,59.0,55.0,109.0,0.7,0.1
1,510.0,42.0,224.0,125.0,139.0,0.0,0.7
2,468.0,40.0,43.0,122.0,240.0,0.5,0.8


#### Network parameters -  2 outputs

In [12]:
input_sequence = Input(shape=(Max_length,))
encod_emb = Embedding(input_dim= aa_vocab_size, output_dim = embedding_size_enc,trainable=True, mask_zero = True)
embedding = encod_emb(input_sequence)

encoder = Bidirectional(GRU(hidden_size_enc, return_sequences=True, return_state = True),
                        merge_mode="concat", weights=None)

encoder_sequence, encoder_final_f, encoder_final_b  = encoder(embedding)

encoder_final = Concatenate(axis=-1)([encoder_final_f, encoder_final_b])



decoder_inputs = Input(shape=(Max_length -1, ))
decoder_inputs_aa = Input(shape=(Max_length, ))

dex=  Embedding(input_dim = dna_vocab_size, output_dim = embedding_size_dec, trainable=True, mask_zero = True)


final_dex= dex(decoder_inputs)
final_dex_aa =  encod_emb(decoder_inputs_aa)


decoder = GRU(2*hidden_size_enc, return_sequences = True, return_state = True)
decoder_aa =  GRU(2*hidden_size_enc_aa, return_sequences = True, return_state = True)

decoder_sequence, decoder_final = decoder(final_dex, initial_state=encoder_final)
decoder_sequence_aa, decoder_final_aa = decoder_aa(final_dex_aa, initial_state=encoder_final)


attn_layer = Attention()
attn_out = attn_layer([decoder_sequence, encoder_sequence])
attn_layer_aa = Attention()
attn_out_aa = attn_layer_aa([decoder_sequence_aa, encoder_sequence])

decoder_concat_input = Concatenate(axis=-1)([decoder_sequence, attn_out]) #decoder_sequence, 
decoder_concat_input_aa = Concatenate(axis=-1)([decoder_sequence_aa, attn_out_aa]) #decoder_sequence,


Intermediate_layer = TimeDistributed(Dense(Dense_layer_size, activation='tanh'))
Intermediate_layer_aa= TimeDistributed(Dense(Dense_layer_size_aa, activation='tanh'))

Intemediate_output = Intermediate_layer(decoder_concat_input) #decoder_concat_input
Intemediate_output_aa = Intermediate_layer_aa(decoder_concat_input_aa) #decoder_concat_input


dropout_layer = Dropout(drop_rate)
dropout_output = dropout_layer(Intemediate_output)

dropout_layer_aa = Dropout(drop_rate_aa)
dropout_output_aa = dropout_layer_aa(Intemediate_output_aa)

dense_layer = TimeDistributed(Dense(dna_vocab_size, activation='softmax'))
logits = dense_layer(dropout_output)

dense_layer_aa = TimeDistributed(Dense(aa_vocab_size, activation='softmax'))
logits_aa = dense_layer_aa(dropout_output_aa)

enc_dec_model = Model([input_sequence, decoder_inputs, decoder_inputs_aa], [logits, logits_aa])

enc_dec_model.compile(loss=sparse_categorical_crossentropy,
              optimizer=Adam(learning_rate = learning_rate),
              metrics=['accuracy','accuracy'])
enc_dec_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1000, 42)  │      1,050 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 1000)      │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_4       │ (None, 999)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ [(None, 1000,     │  1,695,240 │ embedding_2[0][0… │
│ (Bidirectional)     │ 1020), (None,     │            │ not_equal_3[0][0] │
│                     │ 510), (None,      │            │                   │
│                     │ 510)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 999, 224)  │     15,008 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 1020)      │          0 │ bidirectional_1[… │
│ (Concatenate)       │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_4 (GRU)         │ [(None, 999,      │  3,812,760 │ embedding_3[0][0… │
│                     │ 1020), (None,     │            │ concatenate_5[0]… │
│                     │ 1020)]            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 999)       │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_5 (GRU)         │ [(None, 1000,     │  3,255,840 │ embedding_2[1][0… │
│                     │ 1020), (None,     │            │ concatenate_5[0]… │
│                     │ 1020)]            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 1000)      │          0 │ input_layer_5[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_2         │ (None, 999, 1020) │          0 │ gru_4[0][0],      │
│ (Attention)         │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convert_to_tensor_2 │ (None, 999)       │          0 │ not_equal_4[0][0] │
│ (ConvertToTensor)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 1000,      │          0 │ gru_5[0][0],      │
│ (Attention)         │ 1020)             │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 9,330,664 (35.59 MB)

 Trainable params: 9,330,664 (35.59 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
checkpoint_path = "/Users/nickkruyer/PycharmProjects/PichiaCLM/Model_PichiaCLM/2Target_AllData/Arch1_weights_11Mar2026.weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience = 3,
    verbose=0, mode="auto", baseline=None, restore_best_weights=False)
## Train the model
model_results = enc_dec_model.fit([AA_tr[:,1:Max_length+1], Cds_tr[:,0:Max_length-1], AA_tr[:,0:Max_length]], 
                                  [Cds_tr[:,1:Max_length],  AA_tr[:,1:Max_length+1]], 
                                  batch_size= batch_size, 
                                  epochs= epochs, 
                                  validation_split=0.2, callbacks=[cp_callback, early_stop])



Epoch 1/100
